# Customer Churn Prediction: Develop a model to predict customer churn in a subscription-based business.

In [ ]:
import pandas as pd

df = pd.read_csv("customer_churn_dataset.csv")

print(df.head())
print(df.info())
print(df['Churn'].value_counts(normalize=True))

: 

In [ ]:
# remove identifier
df_ml = df.drop(columns=['CustomerID'])

# binary encode gender
df_ml['Gender'] = df_ml['Gender'].map({'Male':1, 'Female':0})

# one-hot encode nominal categories
cat_cols = ['SubscriptionType','PaymentMethod','ContractType']
df_ml = pd.get_dummies(df_ml, columns=cat_cols, drop_first=True)

print(df_ml.head())

In [ ]:
X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Customer Churn Confusion Matrix")
plt.show()


In [ ]:
import pickle, os

folder_path = 'model_artifacts'
os.makedirs(folder_path, exist_ok=True)
save_path = os.path.join(folder_path, 'churn_model.pkl')

package = {
    "model": model,
    "columns": X.columns.tolist()
}

with open(save_path, 'wb') as f:
    pickle.dump(package, f)
